In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
from minsearch import Index

index = Index(
    text_fields={'content':1.0},
    keyword_fields=['filename']
)
index.fit(documents)

In [5]:
question="How does the agentic loop keep calling the model until it stops?"

In [6]:
search_results = index.search(
    question,
    boost_dict={'content':1.0},
  
    num_results=5
)



In [7]:
#search_results[0]

## Homework Questions

In [8]:
len(documents)

72

### 1. How many lesson pages (1 point)

- 24  
- **72**  
- 240  
- 720  

### 2. Indexing and searching (1 point)

- 01-agentic-rag/lessons/03-rag.md  
- **01-agentic-rag/lessons/14-agentic-loop.md**  
- 04-evaluation/lessons/13-llm-as-judge.md  
- 06-best-practices/lessons/02-hybrid-search.md  

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?


We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

In [12]:
from rag_helper import RAGBase
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI


In [16]:
from dotenv import load_dotenv
load_dotenv()

import os

api_key=os.getenv("OPENROUTER_API_KEY")
model="openai/gpt-oss-120b:free"
model='nvidia/nemotron-3-super-120b-a12b:free'

In [22]:
question="what 5x + 3=15 . solve the equation"

In [23]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)
#print(api_key[:15] if api_key else "NO KEY FOUND")
response = client.chat.completions.create(
    model=model
   ,
    messages=[
        {"role": "user", "content": question}
    ]
)

print(response.choices[0].message.content)

To solve  

\[
5x + 3 = 15,
\]

follow these steps:

1. **Subtract 3 from both sides** to isolate the term with \(x\):

\[
5x + 3 - 3 = 15 - 3 \;\Longrightarrow\; 5x = 12.
\]

2. **Divide both sides by 5** to solve for \(x\):

\[
x = \frac{12}{5} = 2.4.
\]

\[
\boxed{x = \frac{12}{5}\ \text{or}\ 2.4}
\]


In [24]:
question

'what 5x + 3=15 . solve the equation'

In [25]:
assistant = RAGBase(index, openai_client, model=model)
assistant.rag(question)

("I don't know.", 5925, 152)

### 3. RAG (1 point)

- 700  
- **7000**  
- 70000  
- 700000  

In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [27]:
len(chunks)

295

In [28]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

### 4. Chunking (1 point)

- 70  
- **295**  
- 1100  
- 4500  

In [29]:
index.fit(chunks)

In [30]:
assistant = RAGBase(index, openai_client, model=model)
assistant.rag(question)

("Idon't know.", 2466, 616)

### 5. RAG with chunking (1 point)

- about the same  
- **3x fewer**  
- 10x fewer  
- 30x fewer  

In [31]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [32]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict = {'content':1.0}
    )

In [33]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [35]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [38]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"ccontent": 1.0}
    )

In [42]:
instructions="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [43]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [44]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [45]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


### 6. Turning it into an agent (1 point)

- 0  
- 4  
- **10**  
- 20  